# Project The counterfeit detection

In [81]:
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from IPython.display import display, HTML
from sklearn import set_config
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    classification_report
)
from sklearn.model_selection import (
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    MinMaxScaler,
    OneHotEncoder,
    RobustScaler,
    StandardScaler,
)
from torch.utils.data import DataLoader
from torchvision import transforms

import config
from src.data_loader import (
    load_data,
    train_val_test_split,
    ImageDataset)
from src.features import ImageFeatureExtractor, SentenceEmbedder
from src.preprocessing import TabularPreprocessor, TextPreprocessor
from src.utils import preview_batch, create_resized_dataset
from src.multimodal import MultiModalFeatureUnion
from src.model import MultiModalClassifier

In [82]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Загрузка данных

In [3]:
data = load_data(config.DATA_CSV)

In [4]:
data.shape

(197198, 44)

In [5]:
data.head(5)

,resolution,brand_name,description,name_rus,commercial_type_name4,rating_1_count,rating_2_count,rating_3_count,rating_4_count,rating_5_count,...,exemplar_returned_count_total30,exemplar_returned_count_total90,exemplar_returned_value_total7,exemplar_returned_value_total30,exemplar_returned_value_total90,item_variety_count,item_available_count,seller_time_alive,item_id,seller_id
id,,,,,,,,,,,,,,,,,,,,,
159385,0,ACTRUM,"Мешки пылесборники для пылесоса PHILIPS, 10 шт...","Мешки для пылесоса PHILIPS TRIATLON, синтетиче...",Пылесборник,6.0,4.0,4.0,3.0,32.0,...,11.0,50.0,730.171845,896.528847,1043.118191,1.0,1.0,1860.0,78312,1218
288616,0,Red Line,Защитная силиконовая крышка обьектива GoPro He...,Защитная крышка Redline на экшн-камеру GoPro (...,Крышка для объектива,NaN,NaN,NaN,NaN,NaN,...,26.0,54.0,993.043882,1137.421611,1188.608000,1.0,1.0,1757.0,141999,1374
108090,0,Talwar Brothers,Плоский медиатор из кости толщиной 0.6 мм<br/>...,Медиатор для гитары Acura GP-PB6,Аксессуар для музыкального инструмента,0.0,0.0,1.0,0.0,1.0,...,16.0,34.0,800.822138,1174.069505,1224.798286,1.0,1.0,1722.0,53306,1448
415607,0,NaN,"Игра Sonic Frontiers для PlayStation 5, русски...","Игра Sonic Frontiers для PlayStation 5, русски...",Видеоигра,NaN,NaN,NaN,NaN,NaN,...,3.0,6.0,0.000000,913.530121,982.789171,3.0,3.0,1692.0,202599,715
332391,0,NaN,Disney Classic Games: Aladdin and The Lion Kin...,"Игра Aladdin and Lion King (PlayStation 4, анг...",Видеоигра,1.0,0.0,0.0,0.0,0.0,...,3.0,6.0,0.000000,913.542170,982.783783,3.0,3.0,1692.0,163725,715


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 197198 entries, 159385 to 104566
Data columns (total 44 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   resolution                       197198 non-null  int64  
 1   brand_name                       116667 non-null  object 
 2   description                      171138 non-null  object 
 3   name_rus                         197198 non-null  object 
 4   commercial_type_name4            197198 non-null  object 
 5   rating_1_count                   47193 non-null   float64
 6   rating_2_count                   47193 non-null   float64
 7   rating_3_count                   47193 non-null   float64
 8   rating_4_count                   47193 non-null   float64
 9   rating_5_count                   47193 non-null   float64
 10  comments_published_count         47193 non-null   float64
 11  photos_published_count           47193 non-null   float64
 12  vi

## Разделение на трейн, валид, тест

In [7]:
# УДАЛИТЬ ПЕРЕД ВЫГРУЗКОЙ
data = data.sample(n=10000, random_state=config.SEED)
data.shape

(10000, 44)

In [8]:
train_df, val_df, test_df, y_train, y_test, y_val = train_val_test_split(
    data, test_size=config.TEST_SIZE, val_size=config.VAL_SIZE, random_state=config.SEED
)

In [9]:
train_df.shape, val_df.shape, test_df.shape

((5000, 43), (2500, 43), (2500, 43))

## Предобработка. Извлечение фич из картинок и описаний

In [10]:
multimodal = MultiModalFeatureUnion()
multimodal

MultiModalFeatureUnion()

In [11]:
multimodal.fit(train_df)

Обнаружено категориальных признаков: 14
Обнаружено числовых признаков: 26
Загрузка модели из кэша model_cache\all-MiniLM-L6-v2
Модель загружена из кэша
Загрузка модели из кэша model_cache\resnet50\model_weights.pth
Препроцессоры обучены


MultiModalFeatureUnion()

In [12]:
comb_train = multimodal.transform(train_df)
comb_train.head(3)

Извлечение эмбеддингов для 5000 текстов...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Получены эмбеддинги: (5000, 384)
...Извлечение признаков из изображений...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Извлечено 5000 признаков размерностью 2048
Извлечены признаки: (5000, 2472)


,comments_published_count,exemplar_accepted_count_total30,exemplar_accepted_count_total7,exemplar_accepted_count_total90,exemplar_returned_count_total30,exemplar_returned_count_total7,exemplar_returned_count_total90,exemplar_returned_value_total30,exemplar_returned_value_total7,exemplar_returned_value_total90,...,img_emb_2038,img_emb_2039,img_emb_2040,img_emb_2041,img_emb_2042,img_emb_2043,img_emb_2044,img_emb_2045,img_emb_2046,img_emb_2047
192916,0.0,1810.0,474.0,5809.0,75.0,30.0,261.0,1132.373215,1054.105798,1251.093806,...,0.285511,0.545595,0.170676,0.381655,1.299194,0.405111,0.638620,0.077434,0.396986,0.209114
84511,0.0,21.0,6.0,62.0,0.0,0.0,2.0,0.000000,0.000000,817.196691,...,0.563394,0.208234,0.500127,0.027349,0.468975,0.023563,0.271265,0.005050,0.046967,0.460535
71384,20.0,71.0,28.0,75.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.666684,0.885570,0.541079,0.328504,0.280005,0.252562,0.098060,0.065360,0.007345,0.266001


In [13]:
comb_val = multimodal.transform(val_df)
comb_val.head(3)

Извлечение эмбеддингов для 2500 текстов...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Получены эмбеддинги: (2500, 384)
...Извлечение признаков из изображений...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Извлечено 2500 признаков размерностью 2048
Извлечены признаки: (2500, 2472)


,comments_published_count,exemplar_accepted_count_total30,exemplar_accepted_count_total7,exemplar_accepted_count_total90,exemplar_returned_count_total30,exemplar_returned_count_total7,exemplar_returned_count_total90,exemplar_returned_value_total30,exemplar_returned_value_total7,exemplar_returned_value_total90,...,img_emb_2038,img_emb_2039,img_emb_2040,img_emb_2041,img_emb_2042,img_emb_2043,img_emb_2044,img_emb_2045,img_emb_2046,img_emb_2047
84331,0.0,1401.0,255.0,3930.0,83.0,18.0,216.0,1234.398811,1095.527345,1320.937040,...,0.678911,0.201299,0.064769,0.024806,0.060782,0.037871,0.288053,0.017216,0.044656,0.261535
159526,0.0,3123.0,561.0,9893.0,254.0,57.0,633.0,1277.155890,1136.731404,1374.119216,...,1.010675,0.471715,0.018562,1.592575,0.083084,0.014491,0.094282,0.016304,1.283071,0.826721
211353,0.0,65.0,28.0,133.0,0.0,0.0,2.0,0.000000,0.000000,745.042368,...,0.329278,0.115866,0.000669,0.185069,0.063794,0.284358,0.055890,0.065554,0.043575,0.128025


In [14]:
comb_test = multimodal.transform(test_df)
comb_test.head(3)

Извлечение эмбеддингов для 2500 текстов...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Получены эмбеддинги: (2500, 384)
...Извлечение признаков из изображений...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Извлечено 2500 признаков размерностью 2048
Извлечены признаки: (2500, 2472)


,comments_published_count,exemplar_accepted_count_total30,exemplar_accepted_count_total7,exemplar_accepted_count_total90,exemplar_returned_count_total30,exemplar_returned_count_total7,exemplar_returned_count_total90,exemplar_returned_value_total30,exemplar_returned_value_total7,exemplar_returned_value_total90,...,img_emb_2038,img_emb_2039,img_emb_2040,img_emb_2041,img_emb_2042,img_emb_2043,img_emb_2044,img_emb_2045,img_emb_2046,img_emb_2047
63782,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.092743,0.190114,0.000000,0.896840,0.776192,0.117221,0.305830,0.180983,0.063902,1.056367
4533,0.0,860.0,246.0,2308.0,36.0,9.0,99.0,1195.540614,1032.930978,1305.306955,...,0.750833,0.089819,0.156811,0.206776,1.087665,0.068027,0.043237,0.598829,1.399496,0.857461
169759,0.0,404.0,65.0,1031.0,4.0,1.0,15.0,713.900842,586.389324,824.632170,...,1.123625,0.347190,0.898974,0.705725,0.016721,0.171066,0.655469,0.030889,0.488578,0.636142


- Получаем список категориальных фич

In [15]:
cat_features = multimodal.get_features()['Category']
cat_features

['item_count_returns7',
 'item_count_fake_returns7',
 'rating_3_count',
 'has_description',
 'rating_2_count',
 'rating_1_count',
 'item_count_fake_returns30',
 'rating_4_count',
 'item_count_fake_returns90',
 'item_count_returns30',
 'item_count_returns90',
 'videos_published_count',
 'has_image',
 'item_count_sales7']

## Обучение модели

In [83]:
model = MultiModalClassifier('catboost', cat_features)
model

MultiModalClassifier(cat_features=['item_count_returns7',
                                   'item_count_fake_returns7', 'rating_3_count',
                                   'has_description', 'rating_2_count',
                                   'rating_1_count',
                                   'item_count_fake_returns30',
                                   'rating_4_count',
                                   'item_count_fake_returns90',
                                   'item_count_returns30',
                                   'item_count_returns90',
                                   'videos_published_count', 'has_image',
                                   'item_count_sales7'],
                     model_params={})

In [84]:
model.fit(comb_train, y_train, comb_val, y_val)

0:	learn: 0.6809885	test: 0.6813558	best: 0.6813558 (0)	total: 195ms	remaining: 1m 37s
1:	learn: 0.6670242	test: 0.6687004	best: 0.6687004 (1)	total: 394ms	remaining: 1m 38s
2:	learn: 0.6571688	test: 0.6598572	best: 0.6598572 (2)	total: 598ms	remaining: 1m 39s
3:	learn: 0.6451235	test: 0.6476640	best: 0.6476640 (3)	total: 795ms	remaining: 1m 38s
4:	learn: 0.6340303	test: 0.6392425	best: 0.6392425 (4)	total: 1.04s	remaining: 1m 42s
5:	learn: 0.6250248	test: 0.6323122	best: 0.6323122 (5)	total: 1.22s	remaining: 1m 40s
6:	learn: 0.6117911	test: 0.6198932	best: 0.6198932 (6)	total: 1.43s	remaining: 1m 40s
7:	learn: 0.6022886	test: 0.6127638	best: 0.6127638 (7)	total: 1.61s	remaining: 1m 39s
8:	learn: 0.5919843	test: 0.6037965	best: 0.6037965 (8)	total: 1.8s	remaining: 1m 38s
9:	learn: 0.5824369	test: 0.5957920	best: 0.5957920 (9)	total: 1.98s	remaining: 1m 37s
10:	learn: 0.5747631	test: 0.5910751	best: 0.5910751 (10)	total: 2.17s	remaining: 1m 36s
11:	learn: 0.5678209	test: 0.5857217	best:

In [85]:
y_pred = model.predict(comb_test)

In [86]:
f1_score(y_test, y_pred)

0.45348837209302323

In [97]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      2335
           1       0.44      0.47      0.45       165

    accuracy                           0.92      2500
   macro avg       0.70      0.71      0.71      2500
weighted avg       0.93      0.92      0.93      2500



In [88]:
print("Train target distribution:", pd.DataFrame(y_train).value_counts())
print("Val target distribution:", pd.DataFrame(y_val).value_counts()) 
print("Test target distribution:", pd.DataFrame(y_test).value_counts())

Train target distribution: 0
0    4669
1     331
Name: count, dtype: int64
Val target distribution: 0
0    2335
1     165
Name: count, dtype: int64
Test target distribution: 0
0    2335
1     165
Name: count, dtype: int64


In [89]:
probas = model.predict_proba(comb_test)
print("Probabilities for class 1:")
print(pd.Series(probas[:, 1]).describe())

Probabilities for class 1:
count    2500.000000
mean        0.238582
std         0.199264
min         0.010000
25%         0.087979
50%         0.169232
75%         0.335532
max         0.947836
dtype: float64


In [90]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"False Positives: {fp}, True Positives: {tp}")

False Positives: 101, True Positives: 78


In [94]:
model.tune_threshold(comb_val, y_val)

Оптимальный порог: 0.609


np.float64(0.609090909090909)